# 📱 Smartphone Data Scraping, Analysis, and Price Prediction Project
## Step 1: Web Scraping and Initial Data Preprocessing

In [ ]:
import numpy as np 
import time          
import pandas as pd
import requests                
from bs4 import BeautifulSoup
import re

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
}  

mydata = []
required_pages = 10

for page_number in range(required_pages):
    current_page = page_number + 1
    
    new_url = f'https://www.amazon.ae/-/ar/s?i=electronics&rh=n%3A15415001031&s=popularity-rank&fs=true&page={current_page}&language=ar'
    
    response = requests.get(new_url, headers=headers)
    print(f"جاري سحب الصفحة {current_page} - حالة الاستجابة: {response.status_code}")
    
    web_page = response.content
    soup = BeautifulSoup(web_page, 'html.parser')
    
    phone_post = soup.find_all('div', {'data-component-type': 's-search-result'})

    if len(phone_post) == 0:
         print(f"تنبيه: لم يتم العثور على أي منتجات في الصفحة {current_page}!")

    for post in phone_post:
        Extracted_data = {
            'full_details': np.NaN,
            'price': np.NaN, 
            'free_delivery_on': np.NaN 
        }

        full_info = post.find('h2', {'class': 'a-size-base-plus'})
        price = post.find('span', {'class': 'a-price-whole'})
        free_delivery_text = post.find(string=re.compile("توصيل مجاني"))        

        if free_delivery_text:
            Extracted_data['free_delivery_on'] = free_delivery_text.strip()
             
        if full_info:
            Extracted_data['full_details'] = full_info.text.strip()
        
        if price:
            Extracted_data['price'] = price.text.strip()

        mydata.append(Extracted_data)
        
    time.sleep(2) 
      
# إنشاء الجدول
df = pd.DataFrame(mydata)

# ==========================================
# مرحلة تنظيف البيانات واستخراج الميزات (Data Preprocessing)
# ==========================================

if not df.empty and 'full_details' in df.columns:
    print("تم سحب البيانات بنجاح، جاري التنظيف واستخراج كافة الميزات...")
    
    brand_pattern = r'(سامسونج|Apple|ابل|شاومي|اونر|Samsung|موتورولا|Motorola|نوثينغ)'
    df['Brand'] = df['full_details'].str.extract(brand_pattern, flags=re.IGNORECASE, expand=False)
    
    model_pattern = r'((?:iPhone|جالكسي|ريدمي|موتورولا|نوثينغ)[^،\-\(]+)'
    df['Exact_Model'] = df['full_details'].str.extract(model_pattern, flags=re.IGNORECASE, expand=False).str.strip()

    df['Storage'] = df['full_details'].str.extract(r'(\d+\s*(?:GB|جيجا|جيجابايت|تيرابايت|TB))', flags=re.IGNORECASE, expand=False)
    
    df['RAM'] = df['full_details'].str.extract(r'((?:RAM|رام)\s*\d+\s*(?:GB|جيجا)?)', flags=re.IGNORECASE, expand=False)
    
    colors_pattern = r'(أسود|أبيض|فضي|رمادي|ذهبي|أزرق|أحمر|أخضر|أصفر|وردي|بنفسجي|برتقالي|كحلي|بني|تيتانيوم|كوبالت|كريمي|نعناعي|Black|White|Silver|Grey|Gray|Gold|Blue|Red|Green|Yellow|Pink|Purple|Titanium|Cream|Mint)'
    df['Color'] = df['full_details'].str.extract(f'({colors_pattern})', flags=re.IGNORECASE, expand=False)

    df['Camera_MP'] = df['full_details'].str.extract(r'(\d+\s*(?:MP|ميجابكسل))', flags=re.IGNORECASE, expand=False)

    
    df['Network_Type'] = df['full_details'].str.extract(r'(5G|LTE|4G)', flags=re.IGNORECASE, expand=False)

    df['AI_Support'] = df['full_details'].str.extract(r'(ذكاء اصطناعي|الذكاء الاصطناعي|ايه اي|AI)', flags=re.IGNORECASE, expand=False)

    df['Battery_Info'] = df['full_details'].str.extract(r'(\d+\s*مللي\s*امبير|\d+\s*واط|عمر\s*بطارية\s*طويل)', flags=re.IGNORECASE, expand=False)

    # ==========================================
    # الترتيب النهائي للأعمدة والحفظ
    # ==========================================
    df = df[['Brand', 'Exact_Model', 'Storage', 'RAM', 'Color', 'Camera_MP', 'Network_Type', 'AI_Support', 'Battery_Info', 'price', 'free_delivery_on', 'full_details']]
    
    df.to_csv('amazon_phones_full_data.csv', index=False, encoding='utf-8-sig')
    print("تم حفظ البيانات وتنظيفها بنجاح في ملف amazon_phones_full_data.csv!")
else:
    print("خطأ: لم يتم سحب أي بيانات. قائمة mydata فارغة.")

جاري سحب الصفحة 1 - حالة الاستجابة: 200
جاري سحب الصفحة 2 - حالة الاستجابة: 200
جاري سحب الصفحة 3 - حالة الاستجابة: 200
جاري سحب الصفحة 4 - حالة الاستجابة: 200
جاري سحب الصفحة 5 - حالة الاستجابة: 200
جاري سحب الصفحة 6 - حالة الاستجابة: 200
جاري سحب الصفحة 7 - حالة الاستجابة: 200
جاري سحب الصفحة 8 - حالة الاستجابة: 200
جاري سحب الصفحة 9 - حالة الاستجابة: 200
جاري سحب الصفحة 10 - حالة الاستجابة: 200
تم سحب البيانات بنجاح، جاري التنظيف والاستخراج...
تم حفظ البيانات وتنظيفها بنجاح في ملف amazon_phones_data.csv!


## 🔍 Step 2: Data Loading and Initial Exploration

In [56]:
import pandas as pd
import numpy as np
df = pd.read_csv('amazon2.csv')
df.head()

,brand,model,Storage,RAM,Color,Price
0,Apple,Iphone 17 Pro Max,256GB,NaN,Silver,4917.0
1,Apple,Iphone 17 Pro,256GB,NaN,Silver,4519.0
2,Samsung,NaN,256GB,12GB,NaN,NaN
3,Samsung,Samsung S26,256GB,12GB,Purple,3248.0
4,Samsung,Samsung A16,NaN,4GB,Black,453.0


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   brand    228 non-null    object 
 1   model    149 non-null    object 
 2   Storage  179 non-null    object 
 3   RAM      146 non-null    object 
 4   Color    168 non-null    object 
 5   Price    212 non-null    float64
dtypes: float64(1), object(5)
memory usage: 11.4+ KB


In [59]:
df.isnull().sum()

brand      12
model      91
Storage    61
RAM        94
Color      72
Price      28
dtype: int64

## 🛠️ Step 3: Smart Missing Value Imputation

In [ ]:
# ── 1. تعويض حسب model أولاً ─────────────────────────────────────────────────
for col in ['Storage', 'RAM']:
    mode_by_model = df.groupby('model')[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x)
    
    df[col] = df[col].fillna(mode_by_model)
# ── 2. ما تبقى → تعويض حسب brand ────────────────────────────────────────────
for col in ['Storage', 'RAM']:
    mode_by_brand = df.groupby('brand')[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x)
    df[col] = df[col].fillna(mode_by_brand)
# ── 3. ما تبقى → القيمة الأكثر شيوعاً في الداتا كاملة ───────────────────────
for col in ['Storage', 'RAM']:
    global_mode = df[col].mode().iloc[0]
    df[col] = df[col].fillna(global_mode)
print("\nبعد التعويض:")
print(f"  storage مفقود: {df['Storage'].isna().sum()}")
print(f"  ram     مفقود: {df['RAM'].isna().sum()}")



بعد التعويض:
  storage مفقود: 0
  ram     مفقود: 0


In [62]:
df.isnull().sum()

brand      12
model      91
Storage     0
RAM         0
Color      72
Price      28
dtype: int64

## 🎨 Step 4: Handling Missing Colors

In [63]:
df['Color'] = df['Color'].fillna('Unknown')


In [64]:
df.isnull().sum()

brand      12
model      91
Storage     0
RAM         0
Color       0
Price      28
dtype: int64

## 🗑️ Step 5: Dropping Unidentified Phones

We dropped the rows missing the `brand`, because it is impossible to process or impute missing values for a completely unidentified phone.

In [65]:
df = df.dropna(subset=['brand']).reset_index(drop=True)


In [66]:
df.isnull().sum()

brand       0
model      79
Storage     0
RAM         0
Color       0
Price      27
dtype: int64

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   brand    228 non-null    object 
 1   model    149 non-null    object 
 2   Storage  228 non-null    object 
 3   RAM      228 non-null    object 
 4   Color    228 non-null    object 
 5   Price    201 non-null    float64
dtypes: float64(1), object(5)
memory usage: 10.8+ KB


## 🎯 Step 6: Dropping Rows with Missing Prices (Target Variable)

We dropped the rows missing the `Price` value. Due to the high variance in smartphone prices, imputing the target variable would introduce significant bias to the model. Furthermore, having the actual ground-truth price is strictly necessary to calculate the model's accuracy and properly evaluate its performance later.

In [68]:
df = df.dropna(subset=['Price']).reset_index(drop=True)


In [69]:
df.isnull().sum()

brand       0
model      69
Storage     0
RAM         0
Color       0
Price       0
dtype: int64

## 🎨 Step 7: Refining the Color Imputation Strategy

In [70]:
df['Color'] = df['Color'].replace('Unknown', np.nan)
print(f"عدد الألوان المفقودة قبل التعويض: {df['Color'].isna().sum()}")
# 2. تعويض اللون المفقود بالقيمة الأكثر تكراراً (Mode) لكل براند (brand)
mode_color_by_brand = df.groupby('brand')['Color'].transform(
    lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
)
df['Color'] = df['Color'].fillna(mode_color_by_brand)


عدد الألوان المفقودة قبل التعويض: 61


## 🧠 Step 8: Comprehensive Handling of Missing Models

In this step, we developed a robust and comprehensive logic utilizing **Antigravity** to ensure that all possible edge cases regarding missing model names are perfectly covered, leaving no room for data gaps:

In [76]:
df.isnull().sum()

brand       0
model      69
Storage     0
RAM         0
Color       0
Price       0
dtype: int64

In [92]:
dataset = pd.read_csv(r"C:\Users\Pc\Desktop\New folder\amazon3.csv")
dataset.head()

,brand,model,Storage,RAM,Color,Price
0,Apple,Iphone 17 Pro Max,256GB,12GB,Silver,4917.0
1,Apple,Iphone 17 Pro,256GB,12GB,Silver,4519.0
2,Samsung,Samsung S26,256GB,12GB,Purple,3248.0
3,Samsung,Samsung A16,256GB,4GB,Black,453.0
4,Apple,Iphone Air,256GB,12GB,Black,3194.0


In [93]:
dataset.isnull().sum()

brand      0
model      0
Storage    0
RAM        0
Color      0
Price      0
dtype: int64

In [94]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   brand    201 non-null    object 
 1   model    201 non-null    object 
 2   Storage  201 non-null    object 
 3   RAM      201 non-null    object 
 4   Color    201 non-null    object 
 5   Price    201 non-null    float64
dtypes: float64(1), object(5)
memory usage: 9.6+ KB


In [95]:
dataset.brand.unique()

array(['Apple', 'Samsung', 'Xiaomi', 'Motorola', 'Nothing', 'Vivo', 'ZTE',
       'Realme', 'Honor', 'Infinix', 'Google', 'G-Tide', 'OnePlus',
       'Tecno', 'Huawei', 'Nokia', 'itel', 'Redmi', 'POCO', 'HMD'],
      dtype=object)

In [96]:
dataset.groupby('model')['Price'].mean()

model
Apple A7          668.000000
G-Tide G81        799.666667
Galaxy A35 5G     898.000000
Google Other     2216.000000
HMD Other         688.000000
                    ...     
iPhone 14        1908.750000
iPhone 15        1831.777778
iPhone 16        3124.428571
itel A5           399.000000
itel S1           899.000000
Name: Price, Length: 83, dtype: float64

In [97]:
dataset.model.unique()

array(['Iphone 17 Pro Max', 'Iphone 17 Pro', 'Samsung S26', 'Samsung A16',
       'Iphone Air', 'Samsung A26 5G', 'Samsung S25', 'Samsung S22',
       'iPhone 15', 'Samsung A57 5G', 'Xiaomi 15', 'Iphone 17',
       'Samsung A06', 'Samsung A56', 'Samsung A17', 'iPhone 16',
       'Samsung A17 5G', 'Motorola T760', 'Nothing A059', 'Vivo X8',
       'ZTE 15R', 'Motorola G56 5G', 'Samsung M07', 'Realme C85',
       'Honor 600 Lite', 'Infinix X6c', 'Google Other', 'Honor X5C',
       'G-Tide G81', 'OnePlus Other', 'ZTE 13s', 'Moto G06', 'Apple A7',
       'Samsung A06 5G', 'iPhone 13', 'Honor 600 5G', 'Honor 400 Lite',
       'Motorola G75 5G', 'Samsung A07', 'ZTE G3', 'Xiaomi 17T',
       'Tecno 200 5G', 'Motorola G06 4G', 'iPhone 14', 'Nothing A059P',
       'Samsung M17', 'Huawei Magic V6 5G', 'Nokia 215', 'Infinix 50X',
       'Xiaomi 16+', 'Nokia 108 4G', 'Samsung M17 5G', 'Tecno X6d',
       'Galaxy A35 5G', 'itel S1', 'Honor X7D', 'Redmi 15C 5G',
       'iPhone 12', 'Samsung A266', '

In [ ]:
dataset = pd.get_dummies(dataset, columns=['Color', 'brand'], drop_first=True  ,dtype=int)    # drop_first=True  Multicollinearity

dataset.head()

,model,Storage,RAM,Price,Color_Blue,Color_Gray,Color_Green,Color_Purple,Color_Red,Color_Silver,...,brand_OnePlus,brand_POCO,brand_Realme,brand_Redmi,brand_Samsung,brand_Tecno,brand_Vivo,brand_Xiaomi,brand_ZTE,brand_itel
0,Iphone 17 Pro Max,256GB,12GB,4917.0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,Iphone 17 Pro,256GB,12GB,4519.0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,Samsung S26,256GB,12GB,3248.0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
3,Samsung A16,256GB,4GB,453.0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,Iphone Air,256GB,12GB,3194.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [99]:
dataset. RAM.unique()

array(['12GB', '4GB', '8GB', '6GB', '16GB', '24GB', '3GB'], dtype=object)

In [100]:
dataset.Storage.unique()


array(['256GB', '512GB', '128GB', '64GB', '1TB'], dtype=object)

## 🧹 Step 9: Standardizing and Converting Capacities to Numeric Values

In this step, we clean the `Storage` and `RAM` columns, converting them from text strings (like "256GB" or "1TB") into pure numerical values in Gigabytes (GB). This transformation is strictly necessary for machine learning algorithms to process the features mathematically.

In [101]:
def clean_to_gb(val):
    val = str(val).strip().upper()
    if 'TB' in val:
        try: return float(val.replace('TB', '').strip()) * 1024
        except: return 0.0
    if 'GB' in val:
        try: return float(val.replace('GB', '').strip())
        except: return 0.0
    return 0.0
dataset['Storage'] = dataset['Storage'].apply(clean_to_gb)
dataset['RAM'] = dataset['RAM'].apply(clean_to_gb)

In [102]:
dataset.Storage.unique()


array([ 256.,  512.,  128.,   64., 1024.])

In [103]:
dataset. RAM.unique()

array([12.,  4.,  8.,  6., 16., 24.,  3.])

In [104]:
dataset['model'].isnull().sum()

0

In [105]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   model           201 non-null    object 
 1   Storage         201 non-null    float64
 2   RAM             201 non-null    float64
 3   Price           201 non-null    float64
 4   Color_Blue      201 non-null    int32  
 5   Color_Gray      201 non-null    int32  
 6   Color_Green     201 non-null    int32  
 7   Color_Purple    201 non-null    int32  
 8   Color_Red       201 non-null    int32  
 9   Color_Silver    201 non-null    int32  
 10  Color_Titanium  201 non-null    int32  
 11  Color_Unknown   201 non-null    int32  
 12  brand_G-Tide    201 non-null    int32  
 13  brand_Google    201 non-null    int32  
 14  brand_HMD       201 non-null    int32  
 15  brand_Honor     201 non-null    int32  
 16  brand_Huawei    201 non-null    int32  
 17  brand_Infinix   201 non-null    int

In [109]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder

X = dataset.drop('Price', axis=1)
y = dataset['Price'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

encoder = TargetEncoder(smooth="auto", target_type="continuous")

X_train["model_encoded"] = encoder.fit_transform(X_train[["model"]], y_train)

X_test["model_encoded"] = encoder.transform(X_test[["model"]])

X_train = X_train.drop('model', axis=1)
X_test = X_test.drop('model', axis=1)

print(X_train[['model_encoded']].head())

     model_encoded
198    2765.503454
38     3760.068827
24     3049.000000
122    1071.074905
196    1646.898438


In [114]:
X_train.head()

,Storage,RAM,Color_Blue,Color_Gray,Color_Green,Color_Purple,Color_Red,Color_Silver,Color_Titanium,Color_Unknown,...,brand_POCO,brand_Realme,brand_Redmi,brand_Samsung,brand_Tecno,brand_Vivo,brand_Xiaomi,brand_ZTE,brand_itel,model_encoded
198,256.0,12.0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,2765.503454
38,512.0,12.0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,3760.068827
24,512.0,12.0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,3049.000000
122,128.0,8.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1071.074905
196,256.0,4.0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1646.898438


In [120]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

mse = np.mean((y_pred - y_test)**2)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)


print(f"(MSE): {mse:.2f} درهم")
print(f"(MAE): {mae:.2f} درهم")
print(f"(RMSE): {rmse:.2f} درهم")
print(f"(R2 Score): {r2:.2f}")

(MSE): 636090.54 درهم
(MAE): 615.90 درهم
(RMSE): 797.55 درهم
(R2 Score): 0.68


In [ ]:
def categorize_phone(model_name):
    # تحويل الاسم إلى نصوص صغيرة لتسهيل البحث
    name = str(model_name).lower()
    
    # 1. فئة الـ Flagship (الهواتف الرائدة والمميزة)
    flagship_keywords = ['pro', 'max', 'ultra', 's22', 's23', 's25', 's26', 'iphone 13', 'iphone 14', 'iphone 15', 'iphone 16', 'iphone 17', 'magic', 'fold', 'air']
    if any(keyword in name for keyword in flagship_keywords):
        return 'Flagship'
    
    # 2. فئة الـ Mid-Range (الهواتف المتوسطة)
    # هواتف سامسونج فئة A العليا (مثل A56, A35) أو فئات T من شاومي
    mid_keywords = ['a5', 'a3', 'm17', '15t', '17t', 'g75', 'g77', 'x300', 'v70']
    if any(keyword in name for keyword in mid_keywords):
        return 'Mid-Range'
    
    # 3. فئة الـ Economy (الهواتف الاقتصادية)
    # أي هاتف لا ينتمي للفئتين السابقتين سيتم اعتباره اقتصادياً
    return 'Economy'

dataset['Phone_Class'] = dataset['model'].apply(categorize_phone)

dataset = pd.pd.get_dummies(dataset, columns=['Phone_Class'], drop_first=True, dtype=int)

dataset = dataset.drop('model', axis=1)

print(dataset.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. تحديد الميزات (X) والهدف (y)
X = dataset.drop(['Price', 'model'], axis=1, errors='ignore')
y = dataset['Price'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

linear_model = LinearRegression()

# 4. تدريب النموذج على البيانات
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

# 6. تقييم الأداء
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== نتائج التقييم ===")
print(f"متوسط الخطأ المطلق (MAE): {mae:.2f} درهم")
print(f"جذر متوسط مربع الخطأ (RMSE): {rmse:.2f} درهم")
print(f"معامل التحديد (R2 Score): {r2:.2f}")

=== نتائج التقييم ===
متوسط الخطأ المطلق (MAE): 655.39 درهم
جذر متوسط مربع الخطأ (RMSE): 840.13 درهم
معامل التحديد (R2 Score): 0.64
